# Qards — una carta cuántica como circuito (Qiskit)

En **Qards** cada carta cuántica es un qubit con dos caras posibles, una por cada
resultado (`|0⟩` y `|1⟩`). La superposición se prepara con una rotación `ry`; al jugar
la carta se **mide** (colapsa) y sale una cara. Dos cartas se **entrelazan** con un CNOT.

Este notebook reproduce las tres piezas con **Qiskit puro**
(`qiskit.quantum_info.Statevector`), sin backend ni Aer: el estado se evalúa en Python y
`sample_counts` lo muestrea. Por eso corre también en el **playground de Qollab (Pyodide)**,
donde no hay wheels nativos.

> Requiere: `python -m pip install -r requirements.txt`.


In [ ]:
import math

from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

SHOTS = 2000


def sample_counts(qc: QuantumCircuit, shots: int = SHOTS) -> dict:
    """Mide el circuito `shots` veces.

    Construye el Statevector (Python puro, sin backend) y lo muestrea.
    Muestrear el estado ES la medición: cada tirada colapsa a una cara.
    """
    return Statevector(qc).sample_counts(shots)


def show(title: str, counts: dict, shots: int = SHOTS) -> None:
    """Imprime una distribución ordenada como porcentajes."""
    print(title)
    for state in sorted(counts):
        n = int(counts[state])
        print(f"  |{state}>  {n:5d}  {100 * n / shots:5.1f}%")

## 1) La carta cuántica: superposición y medición (colapso)

Una carta con caras `|0⟩` y `|1⟩`. La superposición se prepara con `ry(theta)`: la
probabilidad de la cara `|0⟩` es `P(0) = (1+z)/2` con `z = cos(theta)`. Medir el qubit
es **jugar la carta**: colapsa a una de las dos caras.

In [ ]:
# Carta con cara |0> al 80% (z = 0.6 -> P(0) = (1+0.6)/2 = 0.8)
z = 0.6
theta = math.atan2(math.sqrt(1 - z * z), z)

card = QuantumCircuit(1)   # un solo qubit; se mide con `sample_counts`
card.ry(theta, 0)          # prepara la superposición sobre el qubit 0

print(f"P(0) teórica = {(1 + z) / 2:.2f}")
show("Medición de la carta:", sample_counts(card))

## 2) La puerta X: intercambia las caras

La puerta `X` aplica `[[0,1],[1,0]]` y cambia `0 ↔ 1` en la base de medición. **No mide**: la
carta sigue en superposición hasta que se juega, pero con las probabilidades dadas vuelta.

In [ ]:
x_gate = QuantumCircuit(1)
x_gate.ry(theta, 0)   # misma superposición de antes...
x_gate.x(0)           # ...pero con X encima: las caras se dan vuelta
show("Después de X (las caras se intercambian):", sample_counts(x_gate))

## 3) El CNOT: entrelaza dos cartas

Con la carta A en superposición 50/50 y B en `|0⟩`, un CNOT (control A → target B) copia el
valor del control al target: los resultados salen **correlacionados** (solo `|00⟩` y `|11⟩`,
nunca `|01⟩` ni `|10⟩`).

In [ ]:
pair = QuantumCircuit(2)
pair.ry(math.pi / 2, 0)   # carta A en superposición exacta (P(0) = 50%)
pair.cx(0, 1)             # CNOT: control = carta A, target = carta B
show("Par entrelazado (solo |00> y |11>):", sample_counts(pair))

## 4) La fase: H·Z·H ≡ X (la parte genuinamente cuántica)

`Z` no cambia las probabilidades visibles: solo mueve la **fase**. Esa fase se vuelve probabilidad
recién cuando una `H` la vuelve a mezclar, así que `H·Z·H` actúa igual que `X`. Es el lado Qiskit
del test JS del repo (`front/qards/tests/services.test.js`).


In [ ]:
hzh = QuantumCircuit(1); hzh.ry(theta, 0); hzh.h(0); hzh.z(0); hzh.h(0)
xop = QuantumCircuit(1); xop.ry(theta, 0); xop.x(0)
p_hzh = list(Statevector(hzh).probabilities())
p_x = list(Statevector(xop).probabilities())
print("H·Z·H:", [round(float(p), 4) for p in p_hzh])
print("X    :", [round(float(p), 4) for p in p_x])
assert all(abs(a - b) < 1e-9 for a, b in zip(p_hzh, p_x)), "H·Z·H debe coincidir con X"

z_only = QuantumCircuit(1); z_only.ry(theta, 0); z_only.z(0)
p_z = list(Statevector(z_only).probabilities())
assert abs(float(p_z[0]) - (1 + z) / 2) < 1e-9, "Z sola no cambia las probabilidades"
print("Z sola (sin H): P(0) =", round(float(p_z[0]), 4), "-> la fase es invisible hasta que una H la revela")

## 5) El circuito real del juego (los `ops` del motor)

El motor guarda el circuito de cada grupo como una lista plana de operaciones (`ops`);
`back/quantum/qiskit.js` las traduce 1:1 a Qiskit. Este es un grupo entrelazado REAL
(dos cartas + un CNOT).


In [ ]:
OPS_REALES = [["ry", -1.5986421272456757, 0], ["ry", 1.8403951835582866, 1], ["cx", 0, 1]]

def circuito_del_juego(ops):
    qc = QuantumCircuit(2)   # 2 qubits: uno por carta
    for gate, *a in ops:
        if gate == "ry": qc.ry(a[0], a[1])
        elif gate == "cx": qc.cx(a[0], a[1])
        elif gate in ("h", "x", "z"): getattr(qc, gate)(a[0])
        else: raise ValueError(gate)
    return qc

print(circuito_del_juego(OPS_REALES).draw(output="text"))

## 6) IonQ (opcional) — no verificado en este repo

El mismo circuito se puede correr en IonQ con `qiskit-ionq` y un token. **No fue verificado
en este repo** (no hay token ni acceso a Qollab al escribir esto). En el juego, esto lo hace
el adaptador `services/ionq.js` a través del proxy del dev-server.

In [ ]:
# from qiskit import transpile
# from qiskit_ionq import IonQProvider
# provider = IonQProvider(token="TU_IONQ_API_KEY")
# backend = provider.get_backend("ionq_simulator")   # o "ionq_qpu" para hardware
# counts = backend.run(transpile(pair, backend), shots=2000).result().get_counts()
# print(counts)